# D2 — Retrieval Stack & Knowledge Graph

This notebook demonstrates the complete D2 pipeline:

1. **PDF Ingestion** — PyMuPDF extraction, text chunking (500 char / 100 overlap), embedding (BGE), storage to MongoDB + Qdrant
2. **Hybrid Search** — BM25 (sparse) + Dense (Qdrant) + Reciprocal Rank Fusion
3. **Knowledge Graph** — Neo4j with Paper, Author, Topic nodes and WROTE, HAS_TOPIC, CITES relationships
4. **Evaluation** — Recall@K, MRR, nDCG@K metrics

**Prerequisites:** MongoDB, Qdrant, and Neo4j must be running via `docker compose up -d mongodb qdrant neo4j`.

## 0. Setup & Imports

In [ ]:
import sys, os

# Ensure the project root is on the path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

print(f"Working directory: {os.getcwd()}")

---
## 1. PDF Ingestion Pipeline

The ingestion pipeline extracts text from PDFs using PyMuPDF, chunks it into 500-character segments with 100-character overlap, generates 384-dim embeddings using `BAAI/bge-small-en-v1.5`, and stores:
- **MongoDB**: chunk metadata (chunk_id, paper_id, title, authors, text, page range)
- **Qdrant**: dense vector index for cosine similarity search

In [ ]:
from ingest import (
    extract_text_with_pages,
    chunk_text_with_pages,
    embed_texts,
    ingest_directory,
    get_embedding_model,
    EMBEDDING_MODEL,
    EMBEDDING_DIM,
    MONGO_URI,
    MONGO_DB,
    QDRANT_HOST,
    QDRANT_PORT,
)

print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Embedding dim   : {EMBEDDING_DIM}")
print(f"MongoDB URI     : {MONGO_URI}")
print(f"Qdrant          : {QDRANT_HOST}:{QDRANT_PORT}")

### 1.1 Ingest PDFs from the `papers/` directory

If `papers/` is empty, run `python seed_data.py` first to download sample arXiv PDFs.

In [ ]:
import glob

pdfs = glob.glob("papers/*.pdf")
print(f"Found {len(pdfs)} PDFs in papers/:")
for p in pdfs:
    print(f"  - {os.path.basename(p)}")

if pdfs:
    chunks = ingest_directory("papers", batch_size=32)
    print(f"\nIngested {len(chunks)} total chunks.")
else:
    print("\nNo PDFs found. Run `python seed_data.py` first.")

### 1.2 Inspect chunking

Let's look at what a chunked document looks like in MongoDB.

In [ ]:
from pymongo import MongoClient

client = MongoClient(MONGO_URI)
col = client[MONGO_DB]["chunks"]

total = col.count_documents({})
print(f"Total chunks in MongoDB: {total}")

# Show first 3 chunks
print("\n--- Sample chunks ---")
for doc in col.find({}, {"_id": 0}).limit(3):
    print(f"\nchunk_id   : {doc['chunk_id']}")
    print(f"paper_id   : {doc['paper_id']}")
    print(f"title      : {doc.get('title', 'N/A')}")
    print(f"authors    : {doc.get('authors', [])}")
    print(f"pages      : {doc.get('page_start', '?')}-{doc.get('page_end', '?')}")
    print(f"text       : {doc['text'][:200]}...")

### 1.3 Embedding demo

The embedding model is `BAAI/bge-small-en-v1.5` producing 384-dim normalized vectors.

In [ ]:
import numpy as np

model = get_embedding_model()
sample_texts = ["attention mechanism", "knowledge graph", "language model"]
embeddings = embed_texts(sample_texts, model)

print(f"Shape: {embeddings.shape}")
print(f"Dtype: {embeddings.dtype}")
print(f"L2 norm of first vector: {np.linalg.norm(embeddings[0]):.4f}")
print(f"\nFirst 10 dims of 'attention mechanism': {embeddings[0][:10]}")

---
## 2. Hybrid Search (BM25 + Dense + RRF)

The `HybridSearcher` combines:
- **BM25** (sparse lexical matching via `rank-bm25`)
- **Dense** (cosine similarity via Qdrant)
- **RRF** (Reciprocal Rank Fusion with k=60)

Default parameters are loaded from `configs/run_card.yaml` (AutoML winning config: k=11, alpha=0.1154).

In [ ]:
from hybrid_search import HybridSearcher, CONFIG_K, CONFIG_ALPHA

print(f"Config loaded from run_card.yaml:")
print(f"  k (top_k default) : {CONFIG_K}")
print(f"  alpha (BM25 wt)   : {CONFIG_ALPHA}")

searcher = HybridSearcher()

### 2.1 BM25 Search (Sparse)

In [ ]:
query = "attention mechanism in transformers"

bm25_results = searcher.bm25_search(query, top_k=5)
print(f"BM25 results for: '{query}'\n")
for i, r in enumerate(bm25_results, 1):
    print(f"  [{i}] score={r.score:.4f} | {r.title}")
    print(f"      pp. {r.page_start}-{r.page_end} | {r.text[:100]}...\n")

### 2.2 Dense Search (Qdrant)

In [ ]:
dense_results = searcher.dense_search(query, top_k=5)
print(f"Dense results for: '{query}'\n")
for i, r in enumerate(dense_results, 1):
    print(f"  [{i}] score={r.score:.4f} | {r.title}")
    print(f"      pp. {r.page_start}-{r.page_end} | {r.text[:100]}...\n")

### 2.3 Hybrid Search (RRF Fusion)

Reciprocal Rank Fusion merges both ranked lists:

$$\text{RRF}(d) = \sum_{i} \frac{1}{k + \text{rank}_i(d)}, \quad k=60$$

In [ ]:
hybrid_results = searcher.search(query, top_k=5)
print(f"Hybrid (RRF) results for: '{query}'\n")
for i, r in enumerate(hybrid_results, 1):
    print(f"  [{i}] rrf_score={r.score:.6f} | {r.title}")
    print(f"      Citation: {r.citation()}")
    print(f"      Text: {r.text[:120]}...\n")

### 2.4 Filtered Search (paper_ids)

The search methods accept an optional `paper_ids` parameter for graph-guided retrieval (used by D3).

In [ ]:
# Get one paper_id from the corpus to demonstrate filtering
sample_doc = col.find_one({}, {"paper_id": 1, "title": 1, "_id": 0})
if sample_doc:
    target_id = sample_doc["paper_id"]
    print(f"Filtering to paper: {sample_doc.get('title', target_id)}")
    print(f"paper_id: {target_id}\n")

    filtered = searcher.search(query, top_k=5, paper_ids=[target_id])
    print(f"Filtered results ({len(filtered)} chunks):")
    for i, r in enumerate(filtered, 1):
        print(f"  [{i}] {r.score:.6f} | {r.title} | pp. {r.page_start}-{r.page_end}")
else:
    print("No documents in MongoDB. Run seed_data.py first.")

---
## 3. Neo4j Knowledge Graph

The knowledge graph stores three node types and three relationship types:

```
Author --(WROTE)--> Paper --(HAS_TOPIC)--> Topic
                    Paper --(CITES)------> Paper
```

In [ ]:
from graph_build import KnowledgeGraph, populate_from_mongodb, extract_topics

graph = KnowledgeGraph()

### 3.1 Populate graph from MongoDB

In [ ]:
populate_from_mongodb(graph)

stats = graph.stats()
print("\nGraph statistics:")
for key, val in stats.items():
    print(f"  {key}: {val}")

### 3.2 Topic extraction demo

Topics are extracted via keyword matching against 10 categories.

In [ ]:
sample_text = "We propose a new transformer architecture for natural language processing using self-attention."
topics = extract_topics(sample_text)
print(f"Text: {sample_text}")
print(f"Extracted topics: {topics}")

### 3.3 Five Cypher queries

In [ ]:
# Query 1: Topic distribution
print("Topic Distribution:")
for row in graph.count_papers_per_topic():
    print(f"  {row['topic']}: {row['paper_count']} papers")

In [ ]:
# Query 2: Papers by topic
topic = "Natural Language Processing"
papers = graph.find_papers_by_topic(topic)
print(f"\nPapers about '{topic}':")
for p in papers:
    print(f"  - {p['title']}")

In [ ]:
# Query 3: Find related papers (shared topics)
if papers:
    pid = papers[0]["paper_id"]
    related = graph.find_related_papers(pid)
    print(f"\nPapers related to '{papers[0]['title']}':")
    for r in related:
        print(f"  - {r['title']} (shared topics: {r['shared_topics']})")
else:
    print("No papers found for this topic.")

---
## 4. Evaluation Metrics

D2 includes standard IR metrics: **Recall@K**, **MRR**, and **nDCG@K**.

In [ ]:
from hybrid_search import recall_at_k, mrr, ndcg_at_k, run_quick_eval

# Synthetic example
retrieved = ["doc_a", "doc_b", "doc_c", "doc_d", "doc_e"]
relevant = {"doc_c", "doc_e"}

print(f"Retrieved : {retrieved}")
print(f"Relevant  : {relevant}")
print(f"Recall@5  : {recall_at_k(retrieved, relevant, k=5):.3f}")
print(f"MRR       : {mrr(retrieved, relevant):.3f}")
print(f"nDCG@5    : {ndcg_at_k(retrieved, relevant, k=5):.3f}")

### 4.1 Live evaluation against seeded data

In [ ]:
# Build eval queries from papers in MongoDB
pipeline = [
    {"$group": {"_id": "$paper_id", "title": {"$first": "$title"}}},
    {"$limit": 4},
]
paper_docs = list(col.aggregate(pipeline))

eval_queries = []
for doc in paper_docs:
    title = doc.get("title", "")
    if title:
        eval_queries.append({
            "query": title,
            "relevant_ids": [doc["_id"]],
        })

if eval_queries:
    run_quick_eval(searcher, eval_queries)
else:
    print("No papers in MongoDB for evaluation.")

---
## 5. FastAPI Integration

All D2 components are exposed via the FastAPI app in `app.py`:

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/ingest` | POST | Ingest PDFs |
| `/search` | POST | Hybrid BM25+Dense+RRF search |
| `/graph/query` | POST | Raw Cypher queries |
| `/stats` | GET | System statistics |
| `/health` | GET | Service health check |

Start the server with: `uvicorn app:app --host 0.0.0.0 --port 8000 --reload`

In [ ]:
import requests

BASE = "http://localhost:8000"

try:
    health = requests.get(f"{BASE}/health", timeout=3).json()
    print("Health check:")
    for k, v in health.items():
        print(f"  {k}: {v}")

    resp = requests.post(f"{BASE}/search", json={
        "query": "attention mechanism",
        "top_k": 3,
    }, timeout=30).json()
    print(f"\n/search returned {resp['num_results']} results in {resp['elapsed_ms']:.1f}ms")
    for r in resp["results"]:
        print(f"  - {r['citation']}")
except requests.ConnectionError:
    print("API server not running. Start it with: uvicorn app:app --port 8000")

---
## Cleanup

In [ ]:
graph.close()
print("Neo4j connection closed.")
print("\nD2 notebook complete.")